<a href="https://colab.research.google.com/github/tav-rn/TavernTools/blob/main/Prover9_Mace4_Notebook_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction: Prover9/Mace4 with Python (in Colab)



This is a brief (and non-exhaustive) tutorial on using **Python** to run **Prover9/Mace4** inside a **Jupyter-style notebook** (Google Colab).

We assume some familiarity with:
- basic first-order logic (terms, equations, quantifiers),
- the general idea of automated theorem proving / finite model search,
- and elementary Python.

The guiding idea is:

> Prover9 and Mace4 do the grunt work.  
> Python (in a notebook) lets you be clever; helps you generate inputs, run many experiments, parse outputs, and visualize models.

---

## Prover9 and Mace4

[Prover9/Mace4](https://www.cs.unm.edu/~mccune/prover9/) are programs written by William McCune for reasoning about **first-order structures with equality**, including **functions** and **relations**.

Both tools take as input a block of text consisting of **assumptions** and (optionally) **goals**, written in the Prover9/Mace4 syntax (see the [syntax reference](https://www.cs.unm.edu/~mccune/prover9/manual/2009-11A/syntax.html)):
* An *ordinary symbol* is a (maximal) string made from the characters `a-z`, `A-Z`, `0-9`, `$`, and `_`.
* A *special symbol* is a (maximal) string made from the special characters: `{+-*/\^<>=`~?@&|!#';}`.
* A *quoted symbol* is any string enclosed in double quotes.
* The *empty list symbol* is `[]`. This is a special case.

For example, the group axioms can be expressed as:

```x * (y * z) = (x * y) * z.
x * 1 = x.
1 * x = x.
all x exists y (x * y = 1 & y * x = 1).
```

**Note.** Instead of the final sentence (existence of inverses), one can introduce a unary function (for inverse) and use identities such as  
`i(x) * x = 1.` and `x * i(x) = 1.`

## Prover9

**Prover9** attempts to derive a specified **goal** from the **assumptions**, if a proof exists.

A typical use is: supply some axioms (assumptions) and ask Prover9 to prove a target formula (goal). Prover9 outputs a derivation in a clause-based calculus (often as a refutation proof).

For example, if we take as Assumptions the group axioms above, and as Goal the sentence (expressing the identity element 1 is unique in any group):

> ```
>all y (x * y = y & y * x = y) -> x = 1.
>```

Prover9 halts with the following proof:

```
============================== PROOF =================================

% -------- Comments from original proof --------
% Proof 1 at 0.00 (+ 0.01) seconds.
% Length of proof is 7.
% Level of proof is 3.
% Maximum clause weight is 5.000.
% Given clauses 6.


2 (all x (y * x = x & x * y = x)) -> y = 1 # label(non_clause) # label(goal).  [goal].
5 x * 1 = x.  [assumption].
9 c1 * x = x.  [deny(2)].
11 1 != c1.  [deny(2)].
12 c1 != 1.  [copy(11),flip(a)].
17 c1 = 1.  [para(9(a,1),5(a,1)),flip(a)].
18 $F.  [resolve(17,a,12,a)].

============================== end of proof ==========================
```

* Prover9’s internal proofs are typically refutation proofs (proof by contradiction) in a clause-based calculus.
*	The output can be reformatted, expanded/contracted, and also used to generate hints to guide later runs.
*	Even when the proof by contradiction, you can often "read off" a direct argument.

## Mace4

**Mace4** searches for **finite models** of the assumptions, and if a goal is given it tries to find a **countermodel** (a model of the assumptions that refutes the goal). If no goal is provided, Mace4 searches for models of the assumptions.

Mace4 can also:
- search for countermodels of a goal,
- search for all models up to a given size,
- and remove isomorphic duplicates via tools such as `isofilter`.


For example, taking as Assumptions the group axioms above, and as Goal (commutativity):

>```
>x * y = y * x.
>```
   
Mace4 returns the smallest model, i.e., a nonabelian group of mininal cardinality:

```
interpretation( 6, [number = 1,seconds = 0], [
    function(*(_,_), [
        1,0,3,2,5,4,
        0,1,2,3,4,5,
        4,2,1,5,0,3,
        5,3,0,4,1,2,
        2,4,5,1,3,0,
        3,5,4,0,2,1]),
    function(c1, [0]),
    function(c2, [2]),
    function(f1(_), [0,1,2,4,3,5])]).
```
- the big list under *(_ , _) is the operation table (flattened),
-	f1 interprets the unary function symbol (here: an inverse map),
-	and c1, c2 are named constants (often used as witnesses when a goal fails).

For many more options and examples, see the Prover9/Mace4 documentation.

---




## Why Python?

Once you start doing many runs (varying axioms, changing parameters, enumerating structures, filtering outputs), it becomes convenient to let Python do the “glue work”:

- **Build inputs programmatically** (lists of strings, templates, generators),
- **Run Prover9/Mace4 from inside the notebook** and capture output,
- **Parse and clean outputs** (proofs, interpretations),
- **Turn Mace4 models into Python objects** you can compute with,
- **Analyze and visualize models** (tables, graphs, derived operations, checks of extra identities),
- **Post-process proofs** (reformatting, “humanizing” steps, extracting hints).
- **Reformat outputs** for other tools (Prover9 proofs into Lean syntax, or in LaTeX; Mace4 models into UACalc syntax, or Latex tables/tikz diagrams).


In other words: Prover9/Mace4 *find* proofs/models; Python helps you *work with* proofs/models.

---




## Why Jupyter-style notebooks (like Colab)?

A notebook is ideal for a tutorial because it mixes:
- explanations (Markdown),
- experiments (code),
- outputs (proofs, models, plots),
- reusable helper functions you can refine as you go.

Google Colab is especially convenient because it is free, runs in the browser, provides a ready-to-go Python environment, and makes it easy to share a runnable document via a link.
**However**, when inactive for long enough, **the session resets**, so you may lose data/values you've run.

For this reason, many people instead use a Jupyter notebook locally on their device.




## Colab basics

A Colab notebook is a document made of **cells**:
- **Text cells (Markdown)** for explanations, headings, links, equations, etc.
- **Code cells (Python)** that you can run interactively.

When you run code, it executes on a temporary remote machine (“runtime”). The notebook *remembers the state* (variables, imports, files) until you restart the runtime.


---

**Run the current cell**
- Click the **▶** button at the left of the cell, or
- Press **Option + Enter** (Mac) or **Alt + Enter** (Windows/Linux) to run and remain in current cell.
- Press **Shift + Enter** to run and move to next cell.
- Press **Option + Enter** (Mac) or **Alt + Enter** (Windows/Linux) to run and insert new cell below.

> You can run all cells via Menu: **Runtime → Run all**

Test:

In [ ]:
print("hello world")

**The notebook is not “one script”!**
If you run cells out of order, you can get confusing results (variables defined in a later cell, etc.).


# Load `provers` (must run first at startup)

To load the [GitHub repository of Peter Jipsen](https://github.com/jipsen/provers), you run the following cell.

**This must be run at the start of every session** (or reset) on Colab. Only once if on a local Jupyter notebook.


In [ ]:
!wget https://raw.githubusercontent.com/jipsen/provers/refs/heads/master/LoadProver9.py
%run LoadProver9.py

## `prover9.py`

This loads all the functions in the file `prover9.py`, which can be found in this notebooks directory:


```
/content/provers/provers/prover9.py
```

Clicking the link above (or going through the directory on the left side bar, the folder icon) will open this file.

Here, you can inspect how these basic commands work. Why would you do this?
  - To understand what is going on if there is confusion.
  - To adapt, refine, redefine, these functions to your particular case. Simply copy/paste function definitions in a new cell and play.


In what follows, we will highlight and inspect some of these commands, and later on give examples on how to adapt them.

---

## Formula lists

An essential translation from Prover9/Mace4 syntax to that of `provers` is the use of lists of strings as formulas.

For example, recalling from above, the Prover9 syntax for the group axioms:
```prover9
x * (y * z) = (x * y) * z.
x * 1 = x.
1 * x = x.
all x exists y (x * y = 1 & y * x = 1).
```

Will be represented as a list of **strings** (with the `.`) removed.
 - simply copy/paste each line above (without the period `.`) inside quotes `" "` or `' '`

In [ ]:
forumla_list1 = ["x * (y * z) = (x * y) * z",
            "x * 1 = x",
            "1 * x = x",
            "all x exists y (x * y = 1 & y * x = 1)"
            ]

#or with involution
forumla_list2 = ["x * (y * z) = (x * y) * z",
            "x * 1 = x",
            "1 * x = x",
            "i(x) * x = 1",
            "x * i(x) = 1"
            ]
print("Group axioms (without involution in signature): ")
print(forumla_list1)
print("")
print("Group axioms (with involution in signature): ")
print(forumla_list2)

# The basic functions: `prover9()` and  `p9()`



## The `prover9()' function

The principle function used is called `prover9`, which invoke Prover9/Mace4 with lists of formulas and some (limited) options.

```
prover9(assume_list,
        goal_list,
        mace_seconds,
        prover_seconds,
        cardinality,
        options,
        one=False,
        noniso=True,
        info=False,
        params='')
```

**The main inputs:**

* `assume_list`: a list of Prover9 formulas that assumptions

* `goal_list`: a (possibly empty) list of Prover9 formulas that goals

* `mace_seconds`: number of seconds to run Mace4
> the default value is `2` (seconds)


* `prover_seconds`: number of seconds to run Prover9 (**only runs if** mace_seconds < 5)
> the default value is `60` seconds

* `cardinality`:
  - if `None`, search for 1 counter model staring from cardinality 2
  - if an integer `n` (>=2), search for all nonisomorphic models of
cardinality `n`.


Additional inputs:
* `options` -- list of prover9 options (default [], i.e. none)

* `one` -- if `True`, find only one model of given cardinality

* `noniso` -- if `True`, remove isomorphic copies

* `info` -- print input and output of prover9

---

**`prover9()` returns either a list of proofs (via Prover9), or a list of models (via Mace4)**

---

Example of a proof:

In [ ]:
# group axioms as a list, formula_list2 from above
group_ax = ["(x * y) * z = x * (y * z)",
         "x * 1 = x",
         "1 * x = x",
         "x * i(x) = 1",
         "i(x) * x = 1"
         ]
# a identitygoal satisfied by groups
goal = ["i(x * y) = i(y) * i(x)"]

# finds proof
prover9(group_ax, goal)

Example of countermodel:

In [ ]:
# group axioms as a list
group_ax = ["(x * y) * z = x * (y * z)",
         "x * 1 = x",
         "1 * x = x",
         "x * i(x) = 1",
         "i(x) * x = 1"
         ]

# the Goal (list)
comm = ["x * y = y *  x"]

# finds a counter-model of minimal cardinality
prover9(group_ax, comm)

Here is an example of finding a countermodel of a given cardinaltiy using `one = True`.

In [ ]:
# finds a counter-model of cardinality n = 10
prover9(group_ax, comm, cardinality = 10, one = True)

Example of all non-isomorphic models of a given cardinality (note here that the `goal_list` is the empty list `[]`):

In [ ]:
# group axioms as a list
group_ax = ["(x * y) * z = x * (y * z)",
         "x * 1 = x",
         "1 * x = x",
         "x * i(x) = 1",
         "i(x) * x = 1"
         ]
# commutativity
comm = ["x * y = y *  x"]

#abelian groups
abelian_group_ax = group_ax + comm

# finds all abelian groups of size 8
AbG = prover9(abelian_group_ax, [], mace_seconds = 60, cardinality = 8)

Here, `AbG` is a list of 3 models.

In [ ]:
print(AbG)

As with any list, `AbG[0]` is the initial/first entry, so `AbG[i-1]` is the i-th entry.

> **Note** the last entry of any list is obtained by using `-1` as entry.

In [ ]:
print(AbG[0])

In [ ]:
print(AbG[2])

In [ ]:
print(AbG[-1])

It is therefore easy to find all models of a given size with a simple loop:

In [ ]:
#All groups up to size 8

##We first create an empty list,
##and iteratively append to it the list of (noniso) models
##obtained from a run of Mace4 for a given cardinality
groups = []
for n in range(2,9):
  groups.append(prover9(group_ax,[], 60, cardinality = n))

Now `groups` is a list of lists of models.

*   `groups[i]` is the list of all nonisomorphic models of cardinality i+2
*   `groups[i][j]` is the j-th model of cardinality i+2

However, this mismatch of cardinality and index is inconvient and confusing, so we "pad" (entries `0` and `1`) of such lists of lists of models of a given cardinality, call it `L`, so that
  - `L[0] = []`
  - `L[1] = [1]`
  - for n ≥ 2, `L[n]` are the models of cardinality n.

 Redoing the example above:  

In [ ]:
groups = [[],[1]]

for n in range(2,9):
  groups.append(prover9(group_ax,[], 60, cardinality = n))

print(len(groups))

In [ ]:
groups[8][0]

##The `p9()` function

In fact, this behavior is implemented in an alternative function, `p9()`, that is usually the one used in practice/normal use. The inputs are basically the same.

```
p9(assume_list,
        goal_list,
        mace_seconds,
        prover_seconds,
        cardinality,
        options,
        noniso=True,
        info=False,
        params='')
```

The main difference is that `cardinality` accepts a new input:

* if  `cardinality = [n]` for some integer n ≥ 2, search for all nonisomorphic models up to, and including, cardinality n.


> *However, notice there is no longer the input* `one`, so when you want to find a single counterexample of a given cardinality you must use `prover9()`.

---

**`p9()` returns**
 - a **list of proofs** (via Prover9),

 or (via Mace4):
 - a **list of models** if `cardinality = n`, or
 - a **list of lists of models** if `cardinaltiy = [n]`.

---



Revisiting the example above.

In [ ]:
groups = p9(group_ax,[],cardinality = [8])

In [ ]:
print("First model of cardinality 8:")
print(groups[8][0])

---

Using Prover9 is identical.

In [ ]:
p9(group_ax, ["i(x * y) = i(y) * i(x)"])

# The `Proof` class

## `Proof` attributes

In [ ]:
# Remember, proofs are output as a list (as you may have more than one goal)
Ps = p9(group_ax, ["i(x * y) = i(y) * i(x)"],0,60)

# So lets look at its first entry
P = Ps[0]

# P is a proof
print("type(P) == Proof : ", type(P) == Proof)

#What P looks like
print("What the Proof P looks like:")
print(P)

The `Proof` class is very simple, and has the following attributes:

  - `syntax`: a string that indicates what syntax is used for the formulas that represent the proof, e.g. "Prover9".
            
  - `proof` : a list of lists. Each list entry is a list of the form [formula, line_number, [references_to_preceding_lines]]. The references indicate which preceding lines are used in the derivation of the current line.

In [ ]:
#Attributes
print("The syntax:")
print(P.syntax)
print("")
print("The Proof P.proof, which is a a list of lines:")
print(P.proof)
print("")
print("   Each line is a triple, for example:")
print("")
print("The 10th line of proof P.proof[9]:")
print(P.proof[9])
print("   P.proof[9][0] is its line number (integer): ",P.proof[9][0])
print("   P.proof[9][1] is its formula (string): ",P.proof[9][1])
print("   P.proof[9][2] is its references (list): ", P.proof[9][2])

---

## `p9latex()` and `p9lean()`

---
The function `p9latex()` allows one to (naively) translate a list of proofs into LaTeX for better readability (for some common symbols).
  - The first argument must be a list of proofs
  - The second argument `latex` is Boolean
    - if `False` (by default) displays the proof in tex
    - if `True` output the LaTeX code.

In [ ]:
p9latex(Ps)

In [ ]:
p9latex(Ps, latex = True)

---
The function `p9lean()` allows one to (naively) translate a list of proofs into Lean syntax.

In [ ]:
p9lean(Ps)

**Both `p9latex` and `p9lean` are only defined for a small list of specific symbols.** You can always append or change these by copying their source code into a cell, making adjustments, and then running that cell.

# The `Model` class

### `Model` attributes

In [ ]:
# Remember, Mace4 outputs a list models
Gs = p9(group_ax, [],5,0)

# So lets look at its first entry (there should only be one)
G = Gs[0]

#G is a Model
print("type(G) == Model : ", type(G) == Model)

print("")

#What G looks like
print("What the model G looks like:")
print(G)


The `Model` class has a bit more information.

  - `cardinality` : number of elements of the model's base set
  - `index` -- a natural number giving the position of the model
      in a list of models.
  - `operations` : a dictionary of operations on [0..cardinality-1].
      Entries are symbol:table pairs where symbol is a string
      that denotes the operation symbol, e.g. '+', and table is
      an n-dimensional array with entries from [0..cardinality-1].
      n >= 0 is the arity of the operation (not explicitly coded
      but can be computed from the table).
  - `relations` : a dictionary of relations on [0..cardinality-1].
      Entries are symbol:table pairs where symbol is a string
      that denotes the relation symbol, e.g. '<', and table is
      an n-dimensional array with entries from [0,1] (coding
      False/True). Alternatively the table can be an
      (n-2)-dimensional array with entries that are dictionaries
      with keys [0..cardinality-1] and values subsets of [0..cardinality-1],
      given as ordered lists.
      n >= 0 is the arity of the relation (not explicitly coded
      but can be computed from the table).

      ---
      Before we play with examples, let's generate some models with richer structure; in this case, we take bounded involutive lattices (along with the order relation):


In [ ]:
# lattice axioms
lattice_ax = ["x ^ (y ^ z) = (x ^ y) ^ z",
       "x ^ x = x",
       "x ^ y = y ^ x",
       "x v (y v z) = (x v y) v z",
       "x v x = x",
       "x v y = y v x",
       "x ^ (x v y) = x",
       "x v (x ^ y) = x"
       ]
# adding the natural order <=
lattice_w_order = lattice_ax + ["x <= y <-> x ^ y = x"]

# adding an antitone involution '
inv_lattice_ax = lattice_w_order + ["x'' = x", "x <= y -> y' <= x'"]

# add a largest element, a nullary constant T, for good measure.
bdd_inv_lattice_ax = inv_lattice_ax + ["x ^ T = x"]

# All bounded involutiove lattices up to cardinaltiy 6
biLat = p9(bdd_inv_lattice_ax, [],cardinality=[6])


Let's pick a random model and inspect its attributes.

In [ ]:
#look at the model of index 8 (9th member) of cardinality 6
L = biLat[6][8]
L

In [ ]:
print("cardinality (integer):", L.cardinality)
print("")
print("index (integer or None):" , L.index)
print("")
print("operations (dictionary): ", L.operations)
print("")
print("key of operations sybmols (list of strings): ", L.operations.keys())
print("")
print("binary operation ^ (list of lists): ", L.operations["^"])
print("unary operation ' (list of integers): ", L.operations["'"])
print("nullary operation (constant) T (integer between 0,...,cardinality-1): ", L.operations["T"])
print("")
print("relations (dictionary): ", L.relations)
print("key of relation symbols (list of strings):", L.relations.keys())
print("relation <= (list of lists of Boolean values, here as 0/1): ", L.relations["<="])
print("")
print("")
print("The values for 3 v j, for j < cardinality: ", L.operations["v"][3])
print("The value for 3 v 2: ", L.operations["v"][3][2])


## The `show()` command and ordered structures

The `show()` command converts a (list of) model(s), for some certain relations/operations, into a Hasse Diagram to be displayed.

In [ ]:
#Recall L is a lattice with connectives ^ and v
show(L,"v")

In [ ]:
# Recall biLat is a list of list of models
# Here are those models of cardinality 5
K = biLat[5]
show(K,"^")

We can create a simple function that also shows a "class" of models.

In [ ]:
def Show(ls, ops = ["<= v"]):
  """
  ls -- a model, list of model, or list of lists of models
  ops -- a list of operations, as strings, to show
  """
  if type(ls)==Model: ls = [ls]
  if type(ls) == list:
    if type(ls[0])==Model:
      return show(ls, ops)
    elif type(ls[2])==list:
      ls = [ls[i] for i in range(len(ls)) if ls[i] != [] and ls[i] != [1]]
      if type(ls[0][0]) == Model:
        for i in range(len(ls)):
          if ls[i] != []:
            show(ls[i], ops)


In [ ]:
Show(biLat,"^")

In [ ]:
Show(biLat[6],"^")

In [ ]:
Show(biLat[6][0],"^")

### Output for tikz

The fumction `m4tikz()` outputs a tikz translation of the graphs, however there is a type in the `prover9.py` file. Here is a fix to that for it to run.

In [ ]:
def m4tikz(li,symbols="<= v", unaryRel=[]):
  # create a latex/tikz string for a list of digraphs
  st = ""
  for g in m4hasse(li,symbols="<= v", cols=unaryRel):
    st+=dot2tex.dot2tex(str(g), format='tikz', crop=True, figonly=True)+" \\qquad "
  return st

In [ ]:
m4tikz(biLat[6],"^")

## `diagram()`: Diagram of model

Lets start with an example class of models: `Lat` : lattices up to size 6

Some random lattice `L = Lat[6][3]`

In [ ]:
lattice_ax = ["x ^ (y ^ z) = (x ^ y) ^ z",
       "x ^ x = x",
       "x ^ y = y ^ x",
       "x v (y v z) = (x v y) v z",
       "x v x = x",
       "x v y = y v x",
       "x ^ (x v y) = x",
       "x v (x ^ y) = x"
       ]
Lat = p9(lattice_ax, [],60,0,cardinality=[6])

L = Lat[6][3]

print("L:")
print(L)
show(L,"v")

Sometimes its useful to feed (the diagram of) a model into the assumptions. This can be done by transforming the operation/relation tables into a list of identities.

The `diagram` (a `Model` class funtion) outputs this list of identites (i.e., the operation/relation tables written as formulas, line by line).


```
diagram(self,  # Model
        c,     # prefix string,
        s = 0, # Offset, default starting position is 0
        ):

```




Here, `diagram` makes it so elements are distinct. You can relax this condition by taking instead `positive_diagram`.

Below gives the exact diagram with the specified elements.

In [ ]:
L.diagram("")

> Note, the initial lines like `-(0=1)` are reduntant, as numerically assigned elements are always distinct.


In [ ]:
L.positive_diagram("")

Or you can allow flexibility of the assigned elements by making the names variables (usually more useful), those element names take as prefix (to the numeral) the indicated `string` (here take to be `"a"`):

In [ ]:
L.diagram("a")

> Note, the initial lines like, e.g., `-(a0=a1)`, ensures that constant `a0` and the constant `a1` must be assigned to distinct elements from `0`, `...` ,`n-1` (`n` being the cardinality).


In [ ]:
L.positive_diagram("b")

> Here, constants (such as `b0` and `b1`) may be assigned to the same element.

---

### Using the diagrams:

In [ ]:
Ld = p9(L.diagram("a"),[],60,0,cardinality=[L.cardinality])

---
Recall, the model `L` is a member of the class `Lat` of models satisying the formula list:

In [ ]:
lattice_ax = ['x ^ (y ^ z) = (x ^ y) ^ z',
              'x ^ x = x',
              'x ^ y = y ^ x',
              'x v (y v z) = (x v y) v z',
              'x v x = x',
              'x v y = y v x',
              'x ^ (x v y) = x',
              'x v (x ^ y) = x']

One thing we can do is find all models satisfying `lattice_ax`, up to a given cardinality, for which `L` is a substructure (i.e., that `L` embeds into).

In [ ]:
ex = lattice_ax + L.diagram("a")
L_ex = p9(ex,[],60,cardinality=[L.cardinality + 2])

Show(L_ex,"v")

### `inS()`, checking substructures

The function `inS()` checks the model is a subalgebra of another.


```
inS(self,       #Model A
    B,          #Model or cardinalty > A
    info=False
    )
```



For instance, find the collection of (non-isomorphic copies of) `L`'s substrucutres

In [ ]:
S_L = [[A for A in C if A.inS(L)] for C in Lat[2:L.cardinality+1]]

Show(S_L,"v")

### `inH()` and homomorphic images

The function `inH()` checks the model is a homomorphic image of another.


```
inH(self,       #Model A
    B,          #Model or cardinalty > A
    info=False
    )
```



For instance, the collection of `L`'s homomorphic images (up to isomorphism).

In [ ]:
H_L = [[A for A in C if A.inH(L)] for C in Lat[2:L.cardinality+1]]

Show(H_L,"v")

### `product()`, direct products

The function `product()` checks the model is a subalgebra of another.


```
product(self,       #Model A
    B,              #Model B
    info=False
    )
```



In [ ]:
#Product of two lattice
A = Lat[3][0]
B = Lat[4][0]
AB = A.product(B)
show(A,"v")
show(B,"v")
show(AB,"v")

#Some additional tools

## saving models

In [ ]:
import json
def to_dict(self):
    """
    Return a JSON‐serializable dict representing this instance.
    Here, all attributes (int, dict-of-lists, etc.) are already JSON‐friendly,
    so we can just package them up in a new dict.
    """
    return {
        "cardinality": self.cardinality,
        "index": self.index,
        "operations": self.operations
    }

def from_dict(d):
    """
    Given a dict (as returned by to_dict), reconstruct a Model.
    """
    return Model(
        cardinality=d["cardinality"],
        index=d["index"],
        operations=d["operations"]
    )




def save_models(models,file_name = "", download = False):
  models = models[2:]
  if file_name == "":
    from datetime import datetime
    now = datetime.now() # current date and time
    file_name = "models_"+now.strftime("%Y/%m/%d/%H:%M")
  filepath = "/content/" + file_name +".json"
  data = [
      [ to_dict(m) for m in sublist ]
      for sublist in models
  ]
  with open(filepath, "w") as f:
      # no indent ⇒ arrays like [0,1,2,…] stay on one line
      json.dump(data, f)
  if download:
    from google.colab import files
    files.download(f"{file_name}.json")


def load_models(file_name):
    if file_name[-5:] == ".json":
      file_name = file_name[:-5]
    filepath = "/content/" + file_name +".json"
    with open(filepath, "r") as f:
        raw = json.load(f)
    # raw is a list of lists of dicts; rebuild Models
    return [[],[1]] + [
        [ from_dict(d) for d in sublist ]
        for sublist in raw
    ]




## UAcalc out


In [ ]:
import os
import shutil


def uacalc_zip(models, nametag = "A" , folder_name = "UAlgebras", download = True , files_contents = dict()):
  if type(models)==Model: models=[models]
  if type(models[0]) == Model:
    Tot = len(models)
    Tot_mag = len(str(len(models)))
    for i in range(Tot):
      pad = "0"*(len(str(Tot)) - len(str(i)))
      card = models[i].cardinality
      name = f"{nametag}{card}_{pad}{i}"
      text = uacalc_format(models[i],name)
      filename = f"{name}.ua"
      # print(filename)
      files_contents.update({filename : text})
      return files_contents
    return files_contents
  elif type(models) == list and len(models)> 2 and type(models[2])==list: # and type(K[2][0]) == Model:
    for i in range(len(models)):
      if len(models[i])>0 and type(models[i][0]) == Model:
        files_contents = uacalc_zip(models[i], nametag, folder_name, files_contents)
  os.makedirs(folder_name, exist_ok=True)
  for fname, content in files_contents.items():
      path = os.path.join(folder_name, fname)
      with open(path, "w") as f:
          f.write(content)
  shutil.make_archive(folder_name, 'zip', folder_name)
  if download:
    from google.colab import files
    files.download(f"{folder_name}.zip")

In [ ]:
import os
import shutil


def uacalc_zip(models, nametag = "A" , folder_name = "UAlgebras", download = True,flag = True):
  os.makedirs(folder_name, exist_ok=True)
  if type(models)==Model: models=[models]
  if type(models[0]) == Model:
    Tot = len(models)
    Tot_mag = len(str(len(models)))
    for i in range(Tot):
      pad = "0"*(len(str(Tot)) - len(str(i)))
      card = models[i].cardinality
      name = f"{nametag}{card}_{pad}{i}"
      content = uacalc_format(models[i],name)
      fname = f"{name}.ua"
      path = os.path.join(folder_name, fname)
      with open(path, "w") as f:
          f.write(content)
  elif type(models) == list and len(models)> 2 and type(models[2])==list: # and type(K[2][0]) == Model:
    for i in range(len(models)):
      if len(models[i])>0 and type(models[i][0]) == Model:
        uacalc_zip(models[i], nametag, folder_name, download,False)
    flag = True
  #
  if flag:
    shutil.make_archive(folder_name, 'zip', folder_name)
    if download:
      from google.colab import files
      files.download(f"{folder_name}.zip")

## Congruences and Subdirectly Irreducibles

In [ ]:
import contextlib #these are used for removing unwanted printing in calls
import os


def hide_print(command):
  with open(os.devnull, 'w') as devnull: #This is to removing printing from Con
    with contextlib.redirect_stdout(devnull):
      out = command
  return out  # Calls your function without printing to the console

def Con_lat(alg):
  with open(os.devnull, 'w') as devnull: #This is to removing printing from Con
    with contextlib.redirect_stdout(devnull):
          cL = Con(alg)  # Calls your function without printing to the console
  return cL

def isSI(algebra,return_monolith = False):
  def leq(p,q):
    return all(any(x<=y for y in q) for x in p)

  def nontriv(p):
    return any(len(x)>1 for x in p)

  def has_monolith(cL,return_mon): #cL should be a congruence lattice, so input Con(A) for an algebra A
    for p in cL:
      if all(nontriv(p) and (not nontriv(q) or leq(p,q)) for q in cL):
        if return_mon:
          return sorted([list(y) for y in p])
          # return p
        else: return True
        # return p
        # return True
    return False

  with open(os.devnull, 'w') as devnull: #This is to removing printing from Con
    with contextlib.redirect_stdout(devnull):
          cL = Con(algebra)  # Calls your function without printing to the console
  return has_monolith(cL,return_monolith)





#######find subdirectly irreducibles
def find_SI(models):
  if type(models)==Model: models=[models] #if models is not a class of models, but a single algebra A, redfine models as the class [A]
  if type(models[0]) == Model: #if models is a class of algebras
    SI = [A for A in models if isSI(A)]
  else: #if models is a class of classes of algebras
    SI = [[],[]] #Set SIs[0] empty for indexing, and SIs[1] empty for convention that trivial algebras aren't SI
    for i in range(2,len(models)):
      SI.append([A for A in models[i] if isSI(A)])
  return SI

## LaTeX tables display

In [ ]:
import string

def letters(n):
    alphabet = string.ascii_lowercase
    result = []

    # Loop to handle cases when n exceeds the alphabet length
    for i in range(n):
        letter = alphabet[i % 26]
        subscript = i // 26  # Determines if a subscript is needed
        if subscript > 0:
            result.append(f"{letter}_{{{subscript}}}")
        else:
            result.append(letter)

    return result

def neg_names(n):
    alphabet = string.ascii_lowercase
    result = []

    base_n = n if n % 2 == 0 else n - 1

    # Loop to handle cases when base_n exceeds the alphabet length
    for i in range(base_n // 2):
        letter = alphabet[i % 26]
        subscript = i // 26  # Determines if a subscript is needed

        if subscript > 0:
            result.append(f"{letter}_{{{subscript}}}'")
            result.append(f"{letter}_{{{subscript}}}")
        else:
            result.append(f"{letter}' ")
            result.append(letter)


    # If n is odd, add the next unused letter (without a negation)
    if n % 2 != 0:
        next_letter = alphabet[base_n // 2 % 26]
        subscript = base_n // 2 // 26
        if subscript > 0:
            result.append(f"{next_letter}_{{{subscript}}}")
        else:
            result.append(next_letter)

    return result

import sympy as sp
from IPython.display import display, Math


def unary_tab(input_list):
    return [[elem] for elem in input_list]

def tex_table(arr,op = "*",elements = [],tex = False,disp = True):
    if elements == []:
      elements = range(0,len(arr))
    if op == "v":
      op = "\lor "
    if op == "^":
      op = "\land "
    if op == "\ ":
      op = r"\backslash "
    if op == "\\":
      op = r"\backslash "
    if op == "/ ":
      op = "\slash "
    if op == "//":
      op = "\slash "
    if op == "*":
      op = "\cdot "
    if type(arr[0]) == int:
      dim = 0
      arr = unary_tab(arr)
    elif type(arr[0]) == list:
      dim = len(arr[0])
    # Create the LaTeX table header
    latex_str = r"\begin{array}{c|" + "c" * dim + "}\n"

    # Add the first row for the column headers (operands)
    # header_row = op + " & " + " & ".join(map(str, elements)) + r" \\\hline" + "\n"
    header_row = op + " & " + " & ".join(elements) + r" \\\hline" + "\n"
    latex_str += header_row

    # Fill the table rows
    for i, row in enumerate(arr):
        Row = [elements[j] for j in row]
        # latex_row = str(elements[i]) + " & " + " & ".join(map(str, Row)) + r" \\" + "\n"
        latex_row = elements[i] + " & " + " & ".join(Row) + r" \\" + "\n"
        latex_str += latex_row

    # End the LaTeX array
    latex_str += r"\end{array}"

    # Display the LaTeX table
    if disp:
      display(Math(latex_str))
    if tex == True:
      print(f"LaTeX code:\n{latex_str}")
      print(" ")
    return latex_str

def names_with_constants(N,constants,neg_tick = False):
  K = N - len(constants)
  if neg_tick == True:
    Klist = neg_names(K)
  else:
    Klist = letters(K)
  elements = constants + Klist
  return elements



def op_tables_tex(model,ops = [], elements_list = [], tex = False):
  N = model.cardinality
  Tex = []
  # if type(elements_list) == dict:
  #   elements = [elements_list[i] for i in range(N)]
  # else:
  elements = elements_list if not(elements_list == []) else [str(i) for i in range(N)]
  if ops == []:
    ops = list(model.operations.keys())
  for op in ops:
    array = model.operations[op]
    latex_str = tex_table(array,op,elements,tex)
    Tex += [latex_str]

def text2tex(str):
  print(f"LaTeX code:\n{str}")

## Hasse Diagrams


In [ ]:
connectives_dict ={
    "^" : ["m","l"],
    "v" : ["a","r"],
    "*" : ["m","l"],
    "+" : ["a","r"]
}
def additive(op):
  if connectives_dict[op][0] == "a":
    return True
  return False
def edgerev(p,rev=False):
    if rev: return (p[1],p[0])
    return (p[0],p[1])



def bin_ops(model):
  def is_bin_op(arr):
    if type(arr) == list:
      a = arr[0]
      if type(a) == list:
        e = a[0]
        if type(e) == int:
          return True
        else:
          return False
      else:
        return False
    else:
      return False
  return [s for s in model.relations.keys() if is_bin_op(model.relations[s])] + [s for s in model.operations.keys() if is_bin_op(model.operations[s])]

def poset2model_t(A):
    if len(A)==0: raise Error("Can't show Hasse diagram of an empty set")
    k = sorted(A, key = len)
    S = range(len(A))
    if all(type(x)==frozenset for x in k[0]):
      U = union(k[0])
      if all(all(type(y)==frozenset for y in x) and union(x)==U for x in k[1:]): #assume K is a set of partitions
        li = [[all(any(x<=y for y in k[j]) for x in k[i]) for i in S] for j in S]
      else: li = [[k[i]<=k[j] for i in S] for j in S]
    else: li = [[k[i]<=k[j] for i in S] for j in S]
    return Model(cardinality=len(k),relations={"<=":li})



import networkx as nx
from graphviz import Graph
from IPython.display import display_html
def hasse_diagram_t(op,op_s,rel,dual,color_n=[],elements=[],shape_n=[]):
    A = range(len(op))
    if elements == []:
      elements = [str(i) for i in A]
    G = nx.DiGraph()
    # def edgerev(p,rev=False):
    #   if rev: return (p[1],p[0])
    #   return (p[0],p[1])
    # op = op[0]
    if rel:
        G.add_edges_from([(x,y) for x in A for y in A if (op[x][y] if dual else op[y][x]) and x!=y])
        # G.add_edges_from([(x,y) for x in A for y in A if (op[y][x] if dual else op[x][y]) and x!=y])
    else:
        G.add_edges_from([edgerev([x,y], not additive(op_s) if dual else additive(op_s)) for x in A for y in A if op[x][y] == (x if dual else y) and x!=y])
    try:
        G = nx.algorithms.dag.transitive_reduction(G)
    except:
        pass
    P = Graph()
    P.attr('node', shape='circle', width='.3', height='.3', fixedsize='true', fontsize='10')
    # P.attr('node', shape='circle',   fontsize='10')
    for x in A: P.node(str(x), shape = shape_n[x],color= color_n[x],label = elements[x])
    # for x in A: P.node(str(x), color='black',label = elements[x])
    # for x in A: P.node(str(x), color='black')
    P.edges([(str(x[0]),str(x[1])) for x in G.edges])
    return P

def m4diag_t(li,symbols="<= v", color_node="",shape_node = "",name_func = "",show_tables = []):
  # use graphviz to display a mace4 structure as a diagram
  # symbols is a list of binary symbols that define a poset or graph
  # unaryRel is a unary relation symbol that is displayed by red nodes
  i = -1
  sy = symbols.split(" ")
  #print(symbols,"***",sy)
  st = ""
  for x in li:
    if name_func == "": elements =[]
    elif type(name_func) == dict or type(name_func) == list: elements = name_func
    else:
      elements = name_func(x)
    i+=1
    st+=str(i)
    if type(color_node) == str:
      uR = x.relations[color_node] if color_node!="" else ["black"]*x.cardinality
    elif type(color_node) == list  or type(color_node) == dict or type(color_node) == set :
      if len(color_node) == x.cardinality:
        uR = color_node
      else: uR = ["black"]*x.cardinality
    else:
      # print("Either unaryRel is not a relations, or is a list of wrong length")
      uR = color_node(x)

    if type(shape_node) == str:
      iD = ["square" if x.operations[shape_node][i][i] == i else "circle" for i in range(x.cardinality)] if shape_node!="" else ["circle"]*x.cardinality
    elif type(shape_node) == list  or type(shape_node) == dict or type(shape_node) == set :
      if len(color_node) == x.cardinality:
        iD = color_node
      else: iD = ["circle"]*x.cardinality
    else:
      # print("Either unaryRel is not a relations, or is a list of wrong length")
      iD = shape_node(x)
    for s in sy:
            t = s[:-1] if s[-1] == 'd' else s #(s[-1]=='d' or s[-1]=='r')else s
            if t in x.operations.keys():
                st+=hasse_diagram_t(x.operations[t],t,False,s[-1]=='d',uR,elements,iD)._repr_image_svg_xml()+"&nbsp; &nbsp; &nbsp; "
            elif t in x.relations.keys():
                st+=hasse_diagram_t(x.relations[t],t, True, s[-1]=='d',uR,elements,iD)._repr_image_svg_xml()+"&nbsp; &nbsp; &nbsp; "
    st+=" &nbsp; "
  display_html(st,raw=True)


def show_t(K,ops=[],names = "",tables = [],color_node = "", shape_node = ""): # show a list of Mace4 models using graphviz or show a set of subsets or partitions
  if type(K)==Model: K=[K]
  if type(K)==str: K = []
  if type(K)==list and len(K)>0 and type(K[0])==Model:
    def num_elms(m):
      return {i : [str(i),"",""] for i in range(m.cardinality)}
    if names == "": names = num_elms
    def nh(m):
      if (type(names) == list) or (type(names) == dict): return name_format(names)
      else: return name_format(names(m))
    def nt(m):
      if (type(names) == list) or (type(names) == dict): return name_format(names, "tex")
      else: return name_format(names(m),"tex")
    if type(ops) == str:
      st = ops
    else:
      if ops==[]:
        ops = bin_ops(K[0])
      ops=[x.strip() for x in ops]
      st=" ".join(ops)
    if tables:
      for A in K:
        m4diag_t([A],st, color_node, shape_node, name_func = nh, show_tables = tables)
        # m4diag_t([A],st,name_func = names, show_tables = tables)
        for op in tables:
          op_tables_tex(A,[op],nt(A))
    else:
      m4diag_t(K,st, color_node, shape_node, name_func = nh, show_tables = tables)
  elif type(K) == list and len(K)> 2 and type(K[2])==list: # and type(K[2][0]) == Model:
    for i in range(len(K)):
      if len(K[i])>0 and type(K[i][0]) == Model:
        print("-"*20 + f"Cardinality {K[i][0].cardinality}"+ "-"*20)
        show_t(K[i],ops,names,tables, color_node, shape_node)
        #print("-"*20 + f"Cardinality {K[i][0].cardinality}"+ "-"*20)
        #show_t(K[i],ops,names)
  elif type(K)==frozenset: m4diag_t([poset2model_t(K)],"<=d",name_func = names)
  elif type(K)==list: m4diag_t([poset2model(x) for x in K],name_func = names)



In [ ]:
def name_format(names, format = "html"):
  out = []
  for i in range(len(names)):
    let = names[i][0]
    # sub = names[i][1] if names[0][1] else ""
    # sup = names[i][2] if names[0][2] else ""
    sub = "" if len(names[i]) <= 1 else names[i][1]
    sup = "" if len(names[i]) <= 2 else names[i][2]
    if let == "epsilon": let = "&"+let+";" if format == "html" else r'\epsilon'
    elif let == "Delta": let = "&"+let+";" if format == "html" else r'\Delta'
    elif let == "nabla": let = "&"+let+";" if format == "html" else r'\nabla'
    elif let == "top": let = "T" if format == "html" else r'\top'
    if not sub == "": sub = "<sub>"+sub+"</sub>" if format == "html" else "_{"+sub+"}"
    if not sup == "": sup = "<sup>"+sup+"</sup>" if format == "html" else "^{"+sup+"}"
    n = f"<{let}{sub}{sup}>" if format == "html" else let + sub + sup

    out.append(n)
  return out

import string
def block_letters(n):
    alphabet = string.ascii_lowercase
    result = []
    # Loop to handle cases when n exceeds the alphabet length
    for i in range(n):
        letter = alphabet[i % 26]
        subscript = i // 26  # Determines if a subscript is needed
        if subscript > 0:
            #result.append(f"{letter}_{{{subscript}}}")
            result.append(letter * (subscript+1))
        else:
            result.append(letter)
    return result

#name dictionaries

def inv_dict(A):
  N = A.cardinality
  N = range(N)
  n = A.operations["'"]
  names = {}
  E = [e for e in N if n[e] == e]
  k = 0
  for e in E:
    if e == 1:
      names.update({1 : ["1","",""]})
    else:
      k += 1
      e_s = str(k) if len(E) > 1 else ""
      names.update({e : ['epsilon',str(k),""]})
  M = [e for e in N if n[e] != e]
  letters = block_letters(len(M))
  redun = []
  l = 0
  for b in M:
    if not(b in redun):
      redun += [n[b],b]
      if b == 1 or b == n[1]:
        names.update({1 : ["1","",""]})
        names.update({n[1] : ["0","",""]})
      else:
        letter = letters[l]
        names.update({b : [letter,"",""]})
        names.update({n[b] : [letter+"'","",""]})
        l += 1
  return names

def let_dict(A):
  N = A.cardinality
  N = range(N)
  names = {1 : ["1","",""], 0 : ["0","",""]}
  M = [e for e in N if e != 1 and e != 0]
  letters = block_letters(len(M))
  redun = []
  l = 0
  for b in M:
    letter = letters[l]
    names.update({b : [letter,"",""]})
    l += 1
  return names

def idem_rig_dict(A, join = "+"):
  N = range(A.cardinality)
  j = A.operations[join]
  names = {1 : ["1","",""], 0 : ["0","",""]}
  M = [e for e in N if e != 1 and e != 0]
  letters = block_letters(len(M))
  l = 0
  for b in M:
    if all(j[b][x] == b for x in N):
      names.update({b : ["top","",""]})
    else:
      names.update({b : [letters[l],"",""]})
      l += 1
  return names

In [ ]:
#shape/color dicts
def monolith_class_dict(model):
  color_list = ["red",        #0
                "green",      #1
                "blue",       #2
                "purple",     #3
                "pink",       #4
                "brown",      #5
                "cyan",       #6
                "gray",       #7
                "lightblue"   #8
                ]
  mon = isSI(model,True)
  if mon:
    non_triv_eqvc = [eqvc for eqvc in mon if len(eqvc)>1]
    out_dic = {i : "black" for i in range(model.cardinality) if not any(i in eqvc for eqvc in non_triv_eqvc)}
    for j,eqvc in enumerate(non_triv_eqvc):
      for i in eqvc:
        out_dic.update({i : color_list[j]})
  else:
    out_dic = {i : "black" for i in range(model.cardinality)}
  return out_dic


def idrig_shape_dict(model, times = "*", one = 1 , join = "+", bot = 0):
  m = model.operations[times]
  j = model.operations[join]
  N = model.cardinality
  top = [x for x in range(N) if all(j[x][y] == x for y in range(N))][0]
  out = {}
  for x in range(N):
    if m[x][x] == x:
      out.update({x : "doublecircle"})
    elif m[x][x] == top:
      out.update({x : "triangle"})
    elif m[x][x] == bot:
      out.update({x : "invtriangle"})
    elif m[x][x] == one:
      out.update({x : "diamond"})
    else:
      out.update({x : "circle"})
  return out


def mult_rig_dict(A, join = "+",times = "*"):
  n = A.cardinality
  N = range(n)
  j = A.operations[join]
  m = A.operations[times]
  names = {1 : ["1","",""], 0 : ["0","",""]}
  top = [x for x in N if all(j[x][y] == x for y in N)][0]
  if top != 1:
    names.update({top : ["top","",""]})
  M = [e for e in N if e != 1 and e != 0 and e != top]
  def Sg(x):
    x_p = x
    out = [x_p]
    for i in N:
      x_p = m[x_p][x]
      if x_p in out:
        return out
      else: out.append(x_p)
    return sorted(list(set(out)))
  max_Sg = []
  for x in M:
    S_x = Sg(x)
    test_x = [X for X in max_Sg if set(X).issubset(set(S_x))]
    if test_x == []:
      max_Sg.append(S_x)
    else:
      max_Sg = [S_x if set(X).issubset(set(S_x)) else X for X in max_Sg]
  letters = block_letters(len(max_Sg))
  for i,X in enumerate(max_Sg):
    for x in X:
      if not (x in names):
        sup = f"{X.index(x) + 1}" if X.index(x) != 0 else ""
        names.update({x : [letters[i],"",sup]})
  return names